# Attempted mechanistic model

In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import differential_evolution, minimize

import optimization_engine as oe
from optimization_engine import (
    loss_joint, run_model_stim, JOINT_BOUNDS, PARAM_NAMES,
)
from vivo_target import load_vivo_psth
from diagnose_mechanistic import diagnose

# Real in-vivo population PSTH = single source of truth for BOTH the loss and
# the diagnostic plots (replaces the old hand-drawn RULE caricature targets).
bins_plot, mean_vivo_rate, mean_vivo_smooth = load_vivo_psth()
vivo = (bins_plot, mean_vivo_rate, mean_vivo_smooth)
print(f"vivo PSTH: {len(bins_plot)} bins, FR {mean_vivo_rate.min():.0f}..{mean_vivo_rate.max():.0f} Hz")
print("Joint search dims:", len(JOINT_BOUNDS), "(8 geometry + 12 stim/channel)")


In [23]:
# Geometry pulled from the engine (don't redefine bounds locally — they live in
# optimization_engine.py so the loss and the notebook can never drift apart).
N = oe.N
THETA = oe.THETA
T_END = oe.T_END


## Two intrinsic thalamic currents generate the slow component (SC)

Hypothesis: The SC is **not injected** — it emerges from the interaction of two voltage-gated currents that switch on only after the post-flash inhibitory crash hyperpolarizes the cells.

**I_T (T-type Ca²⁺) — sharp post-inhibitory rebound.** Fast activation gate $m_T$ × slow inactivation gate $h_T$:
$$m_T = \sigma\!\big((u-V^{T}_{1/2})/k_T\big),\qquad \tau_{hT}\frac{dh_T}{dt} = h_{T,\infty}(u) - h_T,\qquad h_{T,\infty}=\sigma\!\big(-(u-V^{T}_{1/2})/k_T\big)$$
$$I_T = g_T\, m_T\, h_T\,(E_{Ca}-u)$$
At rest $h_T\approx0$ (inactivated). The crash hyperpolarizes $u$ → $h_T\to1$ (de-inactivation). As the bump recovers through $V^{T}_{1/2}$, $m_T$ turns on while $h_T$ is still high → a transient rebound; then $u$ high → $h_T$ decays → $I_T$ shuts off. $E_{Ca}$ is high-positive so $I_T$ is always depolarizing (no latch).

**I_h (HCN) — slow depolarizing tail.** Additive (not ohmic, to avoid the compressed-scale latch):
$$\tau_h\frac{dm_h}{dt}=m_{h,\infty}(u)-m_h,\qquad I_h = g_h\, m_h$$
with $\tau_h\sim$ hundreds of ms → the slow SC decay.

Full membrane equation:
$$\tau_{E}\frac{du_i}{dt} = -u_i + \textstyle\sum_j W_{ij} r_j - W_{IE} r_I + I_{ext} + I_T + I_h$$

The fast component (FC) is the brief sensory flash $A_{fast}$ (1–5 ms only) — too short to explain the 30–300 ms SC, so the SC **must** come from $I_T$/$I_h$. **Ablation** ($g_T=g_h=0$) removes the SC — the proof it's mechanistic, not curve-fit.

In [ ]:
# SINGLE-STAGE JOINT FIT (all 21 params at once).
# The frozen two-stage split couldn't satisfy baseline + dip + smooth rebound +
# anti-PD silence together: one inhibition knob (W_IE/KAPPA_I) serves all four and
# they conflict, and stage1 committed to it blind to the stim demands. Optimizing
# jointly lets the data MSE penalize the bad regimes (e.g. the late-spike rebound)
# directly. Structural levers (I_baseline<=15, KAPPA_I broad) are pinned via bounds.
# 9 geometry params (incl. the upstream HD drive I_HD) + 12 stim/channel params.
res = differential_evolution(
    loss_joint,
    JOINT_BOUNDS,
    args=(bins_plot, mean_vivo_rate, mean_vivo_smooth),
    seed=42,
    maxiter=400,
    popsize=18,
    tol=1e-6,
    init='sobol',
    mutation=(0.7, 1.9),
    recombination=0.5,
    polish=True,
    disp=True,
    workers=-1,
)
res = minimize(loss_joint, x0=res.x, bounds=JOINT_BOUNDS,
               args=(bins_plot, mean_vivo_rate, mean_vivo_smooth),
               method='L-BFGS-B', options={'maxiter': 400})

params = list(res.x)
stage1_frozen = dict(zip(PARAM_NAMES[:9], params[:9]))
stage2_params = params[9:]
print(f"\nJoint loss = {res.fun:.1f}")
for k, v in zip(PARAM_NAMES, params):
    print(f"  {k:18s} {v:.3f}")


In [ ]:
# OPTIONAL: skip the ~8-min DE above and load the last saved best fit instead
# (written by _run_full_fit.py / the joint cell). Handy for tweaking plots.
import json, pathlib
_p = pathlib.Path("_full_fit_result.json")
if _p.exists():
    saved = json.load(open(_p))["params"]
    params = [saved[k] for k in PARAM_NAMES]
    stage1_frozen = dict(zip(PARAM_NAMES[:9], params[:9]))
    stage2_params = params[9:]
    print("Loaded saved params. R2 was", round(json.load(open(_p)).get("r2", float('nan')), 3))
else:
    print("No saved fit; run the joint cell above.")


In [ ]:
# Full diagnostic: PD vs real PSTH (with ablation), directional traces,
# heatmap, and I_T/I_h channel decomposition.
diagnose(stage1_frozen, stage2_params, vivo=vivo, title="Stage 2 (DE) fit")

In [ ]:
# Quick single-trace inspection (PD cell) if you want it raw.
t_model, rates = run_model_stim(
    *[stage1_frozen[k] for k in PARAM_NAMES[:9]], *stage2_params
)
plt.plot(t_model, rates[:, oe._IDX_0], color="crimson", label="model PD")
plt.plot(bins_plot, mean_vivo_rate, color="0.5", label="in vivo")
plt.xlim(-50, 300); plt.xlabel("time (ms)"); plt.ylabel("Hz"); plt.legend(); plt.show()

### Local polish + final mechanistic fit
L-BFGS-B refines the DE winner; then re-run the full diagnostic. Watch the **ablation** (dotted) and **anti-PD** (navy) traces — they are the evidence the SC is mechanistic and PD-only.

In [ ]:
# Verification summary: the mechanistic checks (not just R2).
t, r = run_model_stim(*[stage1_frozen[k] for k in PARAM_NAMES[:9]], *stage2_params)
pd, anti = r[:, oe._IDX_0], r[:, oe._IDX_180]
# ablation: zero both intrinsic conductances (g_T idx 4, g_h idx 9 of stage2 vec)
abl = list(stage2_params); abl[4] = 0.0; abl[9] = 0.0
_, r0 = run_model_stim(*[stage1_frozen[k] for k in PARAM_NAMES[:9]], *abl)
m = (t > 40) & (t < 300); rec = (t > 200) & (t < 500)
_, ri = oe.run_model_idle(*[stage1_frozen[k] for k in PARAM_NAMES[:9]])
prof = ri[-1, :]; pk = prof.max()
print(f"baseline bump peak   : {pk:.0f} Hz  (target ~40)")
print(f"baseline secondary   : {100*prof[oe.OFF_LOBE].max()/max(pk,.01):.1f}% of peak  (want ~0)")
print(f"FC peak              : {pd[(t>=stage2_params[2])&(t<stage2_params[2]+12)].max():.0f} Hz  (data ~170)")
print(f"dip (shallow)        : {pd[(t>stage2_params[2]+5)&(t<stage2_params[2]+22)].min():.1f} Hz  (data ~25)")
print(f"SC peak (PD)         : {pd[m].max():.0f} Hz @ {t[m][pd[m].argmax()]:.0f} ms  (data ~92 @ ~55)")
print(f"SC ablated (g_T=g_h=0): {r0[:,oe._IDX_0][m].max():.0f} Hz  -> channels generate {100*(1-r0[:,oe._IDX_0][m].max()/max(pd[m].max(),.01)):.0f}% of SC")
print(f"anti-PD SC max       : {anti[m].max():.1f} Hz  (want ~0)")
print(f"recovery PD 200-500ms: {pd[rec].mean():.0f} Hz  (data ~38 -> attractor recovered)")
print(f"I_HD (upstream drive): {stage1_frozen['I_HD']:.2f}")


In [ ]:
# Final mechanistic fit — full diagnostic + emergence evidence.
r2 = diagnose(stage1_frozen, stage2_params, vivo=vivo,
              title="Stage 2 polished — final mechanistic I_T + I_h fit")